# 📄 PDF Document Parser for Product Documentation

## 🎯 Overview
This notebook automates the extraction of text content from PDF product documentation files and stores them in a structured Unity Catalog table. The extracted data can be used for:
- AI-powered customer support chatbots
- Retrieval Augmented Generation (RAG) applications
- Semantic search and vector embeddings
- Product knowledge bases

## 📋 Workflow Steps
1. **Install Dependencies** - Install PyPDF2 library for PDF text extraction
2. **List PDF Files** - Scan the Unity Catalog volume for all PDF documents
3. **Define Extraction Function** - Create a reusable function to extract text from PDFs
4. **Process Documents** - Loop through all PDFs and extract their text content
5. **Save to Unity Catalog** - Store the extracted data in a structured Delta table

## ✅ Prerequisites
- **Source Location**: PDF files stored in Unity Catalog volume
  - Path: `/Volumes/agentic_ai/ecommerce/customer_support/product_docs/`
- **Permissions**: Write access to catalog `agentic_ai`
- **Compute**: Python-enabled Databricks compute (Serverless or cluster)

## 📊 Output
- **Table Name**: `agentic_ai.ecommerce.product_docs`
- **Schema**: 
  - `product_name` (STRING) - Product identifier extracted from filename
  - `product_doc` (STRING) - Complete text content extracted from PDF

## 📦 Step 1: Install Dependencies

We'll use the **PyPDF2** library to extract text from PDF files. 

**Why PyPDF2?**
- Pure Python implementation (no external dependencies)
- Works seamlessly with DBFS and Unity Catalog volumes
- Robust text extraction from standard PDF formats
- Handles multi-page documents efficiently

**Note**: The `%pip install` command installs the package only for this notebook session.

In [0]:
%pip install PyPDF2

## 📂 Step 2: List PDF Files from Unity Catalog Volume

This step scans the Unity Catalog volume to identify all PDF files that need to be processed.

**What we do:**
- Use `dbutils.fs.ls()` to list all files in the volume
- Filter files by `.pdf` extension
- Display the count and names of discovered PDFs

**Volume Path Structure**: `/Volumes/{catalog}/{schema}/{volume}/{path}/`

In [0]:
# Define the Unity Catalog volume path where PDF files are stored
source_path = "/Volumes/agentic_ai/ecommerce/customer_support/product_docs/"

# List all files in the source directory
files = dbutils.fs.ls(source_path)

# Filter to keep only PDF files
pdf_files = [f for f in files if f.name.endswith('.pdf')]

# Display the list of PDF files found
print(f"Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"  - {pdf.name}")

## ⚙️ Step 3: Define Text Extraction Function

Create a reusable `extract_text_from_pdf()` function that handles the entire extraction process.

**Function Workflow:**
1. **Read File** - Opens the PDF file from the volume path
2. **Create PDF Reader** - Initializes PyPDF2 reader object
3. **Iterate Pages** - Loops through all pages in the PDF
4. **Extract Text** - Pulls text content from each page
5. **Return Result** - Returns concatenated text or None if error

**Error Handling**: The function includes try-catch blocks to handle corrupt or unreadable PDFs gracefully.

In [0]:
import PyPDF2
import io

def extract_text_from_pdf(file_path):
    """
    Extract text content from a PDF file stored in DBFS/Unity Catalog volumes.
    
    Args:
        file_path (str): Full path to the PDF file (e.g., /Volumes/catalog/schema/volume/file.pdf)
    
    Returns:
        str: Extracted text content from all pages, or None if extraction fails
    """
    try:
        # Read the PDF file content in binary mode
        # Unity Catalog volumes are accessible via standard file operations
        with open(file_path, "rb") as f:
            pdf_bytes = f.read()
        
        # Create a PDF reader object using PyPDF2
        # BytesIO creates a file-like object from the bytes data
        # Handle both string and bytes types for compatibility
        pdf_reader = PyPDF2.PdfReader(
            io.BytesIO(
                pdf_bytes.encode('latin-1') if isinstance(pdf_bytes, str) else pdf_bytes
            )
        )
        
        # Extract text from all pages in the PDF
        text_content = ""
        for page_num in range(len(pdf_reader.pages)):
            # Get the page object
            page = pdf_reader.pages[page_num]
            
            # Extract text from this page and append to our result
            # Adding newline between pages for better readability
            text_content += page.extract_text() + "\n"
        
        # Return the complete text with leading/trailing whitespace removed
        return text_content.strip()
    
    except Exception as e:
        # Handle any errors (corrupt PDFs, permission issues, etc.)
        print(f"Error extracting text from {file_path}: {str(e)}")
        return None

print("✓ Text extraction function defined successfully!")

## 🔄 Step 4: Process All PDF Documents

This step iterates through each PDF file and builds a structured dataset.

**Processing Logic:**
- Loop through the list of PDF files from Step 2
- Extract product name from filename (remove `.pdf` extension)
- Call the extraction function on each PDF
- Store results in a list of dictionaries with `product_name` and `product_doc` fields
- Track success/failure for each document

**Output**: A Python list containing all successfully extracted product documents.

In [0]:
# Initialize list to store extracted product documentation
product_data = []

# Process each PDF file
# Process each PDF file
for pdf_file in pdf_files:
    print(f"Processing: {pdf_file.name}")
    
    # Extract product name from filename (remove .pdf extension)
    product_name = pdf_file.name.replace('.pdf', '')
    
    # Build full path and extract text content from the PDF
    full_path = f"{source_path}{pdf_file.name}"
    product_doc = extract_text_from_pdf(full_path)
    
    # Add successfully extracted data to our collection
    if product_doc:
        product_data.append({
            'product_name': product_name,
            'product_doc': product_doc
        })
        print(f"  ✓ Successfully extracted {len(product_doc)} characters")
    else:
        print(f"  ✗ Failed to extract text")

print(f"\nTotal documents processed: {len(product_data)}")

## 💾 Step 5: Save to Unity Catalog Table

Convert the extracted data into a Spark DataFrame and persist it to Unity Catalog as a Delta table.

**Process Steps:**
1. **Define Schema** - Create StructType schema with product_name and product_doc fields
2. **Create DataFrame** - Convert Python list to Spark DataFrame with explicit schema
3. **Validate** - Print row count and schema for verification
4. **Write to UC** - Save as managed Delta table in Unity Catalog

**Write Mode**: `overwrite` - Replaces existing table data (use `append` for incremental loads)

**Why Unity Catalog?**
- Centralized data governance and access control
- Easy discovery and sharing across workspaces
- ACID transactions with Delta Lake
- Supports downstream AI/ML workflows

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Define schema for the DataFrame
# - product_name: Name of the product (extracted from filename)
# - product_doc: Full text content extracted from the PDF
schema = StructType([
    StructField("product_name", StringType(), False),
    StructField("product_doc", StringType(), True)
])

# Create Spark DataFrame from the extracted data
# This converts our Python list into a distributed Spark DataFrame
df = spark.createDataFrame(product_data, schema=schema)

# Display the DataFrame schema and row count for validation
print(f"DataFrame created with {df.count()} rows\n")
df.printSchema()

# Write to Unity Catalog table
# Table location: catalog.schema.table
table_name = "agentic_ai.ecommerce.product_docs"
print(f"\nWriting data to table: {table_name}")

# Write the DataFrame to Unity Catalog
# - mode("overwrite"): Replace existing data if table exists
# - overwriteSchema: Allow schema changes if table structure differs
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Successfully created table: {table_name}")

## ✅ Notebook Complete!

### 🎉 What Was Accomplished
- ✓ Installed PyPDF2 library for PDF processing
- ✓ Scanned Unity Catalog volume for PDF files
- ✓ Extracted text content from all product documentation PDFs
- ✓ Created and populated Unity Catalog table: `agentic_ai.ecommerce.product_docs`

---

### 📊 How to Query the Output Table

**Basic Query - View all products:**
```sql
SELECT product_name, LENGTH(product_doc) as doc_length
FROM agentic_ai.ecommerce.product_docs
ORDER BY product_name;
```

**Search for Specific Content:**
```sql
SELECT product_name, product_doc
FROM agentic_ai.ecommerce.product_docs
WHERE product_doc LIKE '%warranty%'
   OR product_doc LIKE '%specifications%';
```

**Get a Specific Product's Documentation:**
```sql
SELECT product_doc
FROM agentic_ai.ecommerce.product_docs
WHERE product_name = 'ProductName';
```

---

### 🚀 Next Steps & Use Cases

**AI/ML Applications:**
- 🤖 Build AI-powered customer support chatbots
- 🔍 Create semantic search with vector embeddings (e.g., using Databricks Vector Search)
- 📚 Feed into RAG (Retrieval Augmented Generation) pipelines
- 🧠 Train product knowledge bases for LLM applications

**Data Pipeline Integration:**
- Schedule this notebook to run automatically when new PDFs are added
- Integrate with downstream feature engineering pipelines
- Create materialized views for specific product categories
- Set up alerts for failed document extractions

**Enhancement Ideas:**
- Add metadata extraction (creation date, author, page count)
- Implement OCR for scanned PDFs
- Extract tables and images separately
- Add data quality checks and validation rules